# AI Workforce Capacity Planning Platform
## Notebook 00 — Project Setup and Enterprise Parameter Framework

**Implementation:** 05 — Enterprise Parameter Framework  
**Project version:** 2.1.0  
**Notebook version:** 3.0.0  

This notebook is the shared platform bootstrap for all downstream notebooks.

It centralizes:

1. project identity,
2. environment configuration,
3. persistent S3 paths,
4. pipeline defaults,
5. forecasting defaults and runtime limits,
6. model-selection defaults,
7. capacity-planning defaults,
8. AI-assistant defaults,
9. reusable parameter validation,
10. storage-access validation.

**Compatibility contract:** Existing global constants remain available so the
validated `02_data_pipeline` notebook continues to work without modification.

In [0]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Any, Mapping, Sequence

## Section 01 — Project Identity

In [0]:
PROJECT_NAME = "AI Workforce Capacity Planning Platform"
PROJECT_KEY = "overtime-capacity-planning"
PROJECT_VERSION = "2.1.0"
PROJECT_SETUP_VERSION = "3.0.0"
ENVIRONMENT = "development"
PROJECT_OWNER = "Issouf KABRE"

PROJECT_INITIALIZED_AT_UTC = datetime.now(timezone.utc)

PROJECT_CONFIG: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "project_key": PROJECT_KEY,
    "project_version": PROJECT_VERSION,
    "project_setup_version": PROJECT_SETUP_VERSION,
    "environment": ENVIRONMENT,
    "project_owner": PROJECT_OWNER,
}

## Section 02 — Persistent Storage Configuration

Bronze remains the first persistent data layer.
Source files may exist temporarily during acquisition, but no persistent
Landing copy is required for the current prototype architecture.

In [0]:
S3_BUCKET = "issouf-data-lake"
PROJECT_ROOT = f"s3a://{S3_BUCKET}/{PROJECT_KEY}"

BRONZE_ROOT = f"{PROJECT_ROOT}/bronze"
SILVER_ROOT = f"{PROJECT_ROOT}/silver"
GOLD_ROOT = f"{PROJECT_ROOT}/gold"

METADATA_ROOT = f"{PROJECT_ROOT}/metadata"
REGISTRY_ROOT = f"{PROJECT_ROOT}/registry"
MANIFEST_ROOT = f"{METADATA_ROOT}/manifests"
VALIDATION_ROOT = f"{METADATA_ROOT}/validation"
PIPELINE_LOG_ROOT = f"{METADATA_ROOT}/pipeline_logs"

MODEL_ROOT = f"{PROJECT_ROOT}/models"
FORECAST_ROOT = f"{PROJECT_ROOT}/forecasts"
DECISION_ROOT = f"{PROJECT_ROOT}/decisions"
REPORT_ROOT = f"{PROJECT_ROOT}/reports"

DATASET_REGISTRY_PATH = f"{REGISTRY_ROOT}/dataset_registry"

STORAGE_CONFIG: dict[str, str] = {
    "s3_bucket": S3_BUCKET,
    "project_root": PROJECT_ROOT,
    "bronze_root": BRONZE_ROOT,
    "silver_root": SILVER_ROOT,
    "gold_root": GOLD_ROOT,
    "metadata_root": METADATA_ROOT,
    "registry_root": REGISTRY_ROOT,
    "dataset_registry_path": DATASET_REGISTRY_PATH,
    "manifest_root": MANIFEST_ROOT,
    "validation_root": VALIDATION_ROOT,
    "pipeline_log_root": PIPELINE_LOG_ROOT,
    "model_root": MODEL_ROOT,
    "forecast_root": FORECAST_ROOT,
    "decision_root": DECISION_ROOT,
    "report_root": REPORT_ROOT,
}

# Backward-compatible shared path map.
PROJECT_PATHS: dict[str, str] = {
    "project_root": PROJECT_ROOT,
    "bronze": BRONZE_ROOT,
    "silver": SILVER_ROOT,
    "gold": GOLD_ROOT,
    "metadata": METADATA_ROOT,
    "registry": REGISTRY_ROOT,
    "dataset_registry": DATASET_REGISTRY_PATH,
    "manifests": MANIFEST_ROOT,
    "validation": VALIDATION_ROOT,
    "pipeline_logs": PIPELINE_LOG_ROOT,
    "models": MODEL_ROOT,
    "forecasts": FORECAST_ROOT,
    "decisions": DECISION_ROOT,
    "reports": REPORT_ROOT,
}

## Section 03 — Data-Pipeline Parameters

In [0]:
PIPELINE_CONFIG: dict[str, Any] = {
    "pipeline_name": "enterprise-workforce-data-foundation",
    "pipeline_version": "3.1.0",
    "write_mode": "overwrite",
    "parquet_compression": "snappy",
    "enable_data_quality_checks": True,
    "save_execution_log": True,
    "fail_on_empty_dataset": True,
    "fail_on_row_count_mismatch": True,
    "fail_on_duplicate_business_keys": True,
}

# Backward-compatible constants for downstream notebooks.
PIPELINE_NAME = PIPELINE_CONFIG["pipeline_name"]
PIPELINE_VERSION = PIPELINE_CONFIG["pipeline_version"]
PARQUET_WRITE_MODE = PIPELINE_CONFIG["write_mode"]
PARQUET_COMPRESSION = PIPELINE_CONFIG["parquet_compression"]

## Section 04 — Forecast Parameters

The active forecast horizon is a runtime parameter.

This configuration defines:

- a default value,
- accepted boundaries,
- validation behavior,
- forecasting metadata.

It does **not** hard-code the business to one-day, seven-day, or
fourteen-day forecasting.

In [0]:
FORECAST_CONFIG: dict[str, Any] = {
    "default_horizon_days": 14,
    "minimum_horizon_days": 1,
    "maximum_horizon_days": 90,
    "frequency": "daily",
    "date_column": "order_date",
    "target_column": "workload_units",
    "validation_horizon_days": 28,
    "minimum_training_rows": 180,
    "random_seed": 42,
    "retrain_model": True,
    "save_model": True,
    "confidence_level": 0.95,
}

DEFAULT_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["default_horizon_days"]
MINIMUM_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["minimum_horizon_days"]
MAXIMUM_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["maximum_horizon_days"]

## Section 05 — Model Parameters

In [0]:
MODEL_CONFIG: dict[str, Any] = {
    "candidate_models": [
        "seasonal_naive",
        "linear_regression",
        "random_forest",
        "gradient_boosting",
    ],
    "primary_metric": "mae",
    "secondary_metrics": [
        "rmse",
        "mape",
        "wape",
    ],
    "selection_strategy": "lowest_validation_mae",
    "model_artifact_format": "mlflow",
    "register_best_model": True,
}

## Section 06 — Capacity-Planning Parameters

These values are defaults for the public-data prototype.
Operational production values must be confirmed with warehouse
stakeholders before real-world deployment.

In [0]:
CAPACITY_CONFIG: dict[str, Any] = {
    "standard_shift_hours": 10.0,
    "maximum_daily_hours": 10.0,
    "productivity_unit": "order_lines_per_associate_hour",
    "voluntary_overtime_enabled": True,
    "mandatory_overtime_enabled": True,
    "weekend_enabled": True,
    "holiday_adjustment_enabled": True,
    "voluntary_capacity_gap_ratio": 0.50,
    "mandatory_capacity_gap_ratio": 1.00,
    "minimum_overtime_hours": 1.0,
    "maximum_overtime_hours": 4.0,
}

## Section 07 — AI-Assistant Parameters

Secrets and credentials must never be stored in this configuration.

In [0]:
AI_CONFIG: dict[str, Any] = {
    "assistant_enabled": True,
    "include_forecast_context": True,
    "include_capacity_context": True,
    "include_decision_explanation": True,
    "maximum_context_records": 30,
    "response_style": "operations_management",
    "human_review_required": True,
}

## Section 08 — Shared Validation Utilities

In [0]:
def validate_required_keys(
    *,
    config_name: str,
    config: Mapping[str, Any],
    required_keys: Sequence[str],
) -> None:
    """Validate that required configuration keys exist and are non-null."""

    missing_keys = [
        key
        for key in required_keys
        if key not in config or config[key] is None
    ]

    if missing_keys:
        raise ValueError(
            f"{config_name} is missing required keys: "
            f"{sorted(missing_keys)}"
        )


def validate_integer_range(
    *,
    parameter_name: str,
    value: int,
    minimum: int,
    maximum: int,
) -> int:
    """Validate an integer runtime parameter and return the accepted value."""

    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(
            f"{parameter_name} must be an integer; "
            f"received {type(value).__name__}."
        )

    if not minimum <= value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between {minimum} and {maximum}; "
            f"received {value}."
        )

    return value


def validate_numeric_range(
    *,
    parameter_name: str,
    value: float,
    minimum: float,
    maximum: float,
) -> float:
    """Validate a numeric parameter and return the accepted value."""

    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(
            f"{parameter_name} must be numeric; "
            f"received {type(value).__name__}."
        )

    accepted_value = float(value)

    if not minimum <= accepted_value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between {minimum} and {maximum}; "
            f"received {accepted_value}."
        )

    return accepted_value


def resolve_forecast_horizon(
    requested_horizon_days: int | None = None,
) -> int:
    """
    Resolve and validate the active runtime forecast horizon.

    When no runtime value is supplied, the configured default is used.
    """

    active_horizon = (
        DEFAULT_FORECAST_HORIZON_DAYS
        if requested_horizon_days is None
        else requested_horizon_days
    )

    return validate_integer_range(
        parameter_name="forecast_horizon_days",
        value=active_horizon,
        minimum=MINIMUM_FORECAST_HORIZON_DAYS,
        maximum=MAXIMUM_FORECAST_HORIZON_DAYS,
    )


def validate_platform_configuration() -> None:
    """Validate all shared platform configuration contracts."""

    validate_required_keys(
        config_name="PROJECT_CONFIG",
        config=PROJECT_CONFIG,
        required_keys=[
            "project_name",
            "project_key",
            "project_version",
            "environment",
        ],
    )

    validate_required_keys(
        config_name="STORAGE_CONFIG",
        config=STORAGE_CONFIG,
        required_keys=[
            "project_root",
            "bronze_root",
            "silver_root",
            "gold_root",
            "metadata_root",
            "model_root",
            "forecast_root",
            "decision_root",
        ],
    )

    validate_required_keys(
        config_name="PIPELINE_CONFIG",
        config=PIPELINE_CONFIG,
        required_keys=[
            "pipeline_name",
            "pipeline_version",
            "write_mode",
            "parquet_compression",
        ],
    )

    validate_required_keys(
        config_name="FORECAST_CONFIG",
        config=FORECAST_CONFIG,
        required_keys=[
            "default_horizon_days",
            "minimum_horizon_days",
            "maximum_horizon_days",
            "frequency",
            "date_column",
            "target_column",
        ],
    )

    validate_required_keys(
        config_name="MODEL_CONFIG",
        config=MODEL_CONFIG,
        required_keys=[
            "candidate_models",
            "primary_metric",
            "selection_strategy",
        ],
    )

    validate_required_keys(
        config_name="CAPACITY_CONFIG",
        config=CAPACITY_CONFIG,
        required_keys=[
            "standard_shift_hours",
            "productivity_unit",
            "human_review_required",
        ]
        if "human_review_required" in CAPACITY_CONFIG
        else [
            "standard_shift_hours",
            "productivity_unit",
            "voluntary_overtime_enabled",
            "mandatory_overtime_enabled",
        ],
    )

    validate_required_keys(
        config_name="AI_CONFIG",
        config=AI_CONFIG,
        required_keys=[
            "assistant_enabled",
            "human_review_required",
        ],
    )

    minimum_horizon = FORECAST_CONFIG["minimum_horizon_days"]
    default_horizon = FORECAST_CONFIG["default_horizon_days"]
    maximum_horizon = FORECAST_CONFIG["maximum_horizon_days"]

    if not minimum_horizon <= default_horizon <= maximum_horizon:
        raise ValueError(
            "FORECAST_CONFIG horizon contract is invalid: "
            "minimum <= default <= maximum is required."
        )

    validate_numeric_range(
        parameter_name="confidence_level",
        value=FORECAST_CONFIG["confidence_level"],
        minimum=0.50,
        maximum=0.999,
    )

    validate_numeric_range(
        parameter_name="standard_shift_hours",
        value=CAPACITY_CONFIG["standard_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

## Section 09 — Runtime Parameter Contract

Notebook `03_forecasting_engine` will supply the actual horizon at
execution time, typically through a Databricks widget.

Notebook 00 only provides the default and validation contract.

In [0]:
ACTIVE_FORECAST_HORIZON_DAYS = resolve_forecast_horizon()

RUNTIME_CONFIG: dict[str, Any] = {
    "forecast_horizon_days": ACTIVE_FORECAST_HORIZON_DAYS,
    "environment": ENVIRONMENT,
    "initialized_at_utc": PROJECT_INITIALIZED_AT_UTC,
}

## Section 10 — Platform Configuration Validation

In [0]:
validate_platform_configuration()
CONFIGURATION_STATUS = "PASSED"

## Section 11 — Storage Access Validation

This check confirms that Databricks can access the project root.
It does not create temporary validation data.

In [0]:
try:
    dbutils.fs.ls(PROJECT_ROOT)
    STORAGE_CONNECTION_OK = True
    STORAGE_STATUS = "PASSED"
except Exception as exc:
    STORAGE_CONNECTION_OK = False
    STORAGE_STATUS = "FAILED"
    raise RuntimeError(
        f"Unable to access project storage: {PROJECT_ROOT}"
    ) from exc

## Section 12 — Execution Summary

In [0]:
print("=" * 80)
print(PROJECT_NAME.upper())
print("=" * 80)
print(f"Project version          : {PROJECT_VERSION}")
print(f"Setup notebook version   : {PROJECT_SETUP_VERSION}")
print(f"Environment              : {ENVIRONMENT}")
print(f"Project root             : {PROJECT_ROOT}")
print(f"Configuration status     : {CONFIGURATION_STATUS}")
print(f"Storage status           : {STORAGE_STATUS}")
print(f"Default forecast horizon : {DEFAULT_FORECAST_HORIZON_DAYS} days")
print(
    "Allowed horizon range   : "
    f"{MINIMUM_FORECAST_HORIZON_DAYS}–"
    f"{MAXIMUM_FORECAST_HORIZON_DAYS} days"
)
print(f"Runtime status           : READY")
print("=" * 80)

AI WORKFORCE CAPACITY PLANNING PLATFORM
Project version          : 2.1.0
Setup notebook version   : 3.0.0
Environment              : development
Project root             : s3a://issouf-data-lake/overtime-capacity-planning
Configuration status     : PASSED
Storage status           : PASSED
Default forecast horizon : 14 days
Allowed horizon range   : 1–90 days
Runtime status           : READY
